# 3. 临床模型

复用影像模型的同一患者集合、结局和 train/test 划分。分类编码、缺失值填补、标准化均在训练折拟合。

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import joblib
import ov_radiomics as rad

# Run notebooks in order, with examples/ as the working directory.
output = Path("output/notebook_demo")

In [ ]:
clinical = pd.read_csv(output / "clinical.csv")
clinical_result = rad.train_models(
    clinical, target="label", features=["age", "sex", "marker"],
    continuous=["age", "marker"], categorical=["sex"],
    scale="standard", models=["LR", "RandomForest"],
    cv=3, random_state=42,
    model_params={"RandomForest": {"n_estimators": 50}},
)
clinical_result.save(output / "clinical")
clinical_result.cv_results

`scale=None` 可关闭数值变量标准化。分类变量采用 one-hot 编码；测试时未知水平按编码器的 `handle_unknown="ignore"` 处理。

`.oof_predictions` 的每一行来自未使用该样本训练的基础模型，将用于下一步融合。

In [ ]:
clinical_result.oof_predictions.head()

In [ ]:
rad.plot_cv_results(clinical_result.cv_results)
metrics = rad.evaluate(clinical_result, threshold=0.5)
rad.plot_metrics(metrics, metric="auc")
plt.show()
metrics

In [ ]:
test = clinical_result.predictions.query('group == "test" and model == "LR"')
rad.plot_roc(test.y_true, test.probability)
rad.plot_confusion_matrix(test.y_true, test.probability, threshold=0.5)
plt.show()